In [200]:
import uproot
import pandas as pd
import numpy as np
import awkward as ak
import os
import json
import ROOT

In [201]:
basedir = os.path.join(os.environ.get('ttH_yy_DIR'), 'final')

process = {
    'ttHyy' : { 'sample_list' : ['mgp8_pp_tth01j_5f_haa'], 
               'label' : r'$ttH \rightarrow \gamma\gamma$'},
    'yy_jets' : { 'sample_list' : ['mgp8_pp_jjaa_5f'], 
               'label' : r'$\gamma\gamma$ + jets'},
    'ttyy' : { 'sample_list' : ['mgp8_pp_ttaa_semilep_5f_100TeV'], 
               'label' : '$tt\gamma\gamma$'}
}

selection = {
    'sel0_nocuts' : "All events",
    'sel1_bjets' : '$\geq 2$ $b$-jets',
    'sel2_photons' : '$\geq 2$ photons',
    'sel3_myy_window' : "120 < $m_{\gamma\gamma}$ < 130 GeV"
}

variables = ['weight']

process_infos = "/eos/experiment/fcc/hh/utils/FCCDicts/FCChh_procDict_fcc_v06_II.json"

lumi = 3e7 # 3 * 10^7 pb-1 = 3 ab-1

In [202]:
with open(process_infos, 'r') as f :
    process_dict = json.load(f)

In [203]:
df = {}
sow = {}

In [204]:
for p in process.keys() :
    df[p] = {}
    for s in selection.keys() :
        df1 = []
        sow[p] = []
        for sample in process[p]['sample_list'] : 
            inputfile = os.path.join(basedir, sample+"_"+s+".root")
            # get sum of weights
            f = ROOT.TFile.Open(inputfile)
            sow[p].append(f.Get("SumOfWeights").GetVal())
            f.Close()
            # get sample
            with uproot.open(inputfile) as f :
                df1.append(ak.to_dataframe(f['events'].arrays(expressions=variables, library='ak')))
                df1[-1]["weight"] = df1[-1]["weight"]/sow[p][-1]*process_dict[sample]['crossSection']*process_dict[sample]['kfactor']*process_dict[sample]['matchingEfficiency']
        df[p][s] = pd.concat(df1, copy=True, ignore_index=True)

In [205]:
my_entries = {
    "Selection" : []
}

In [206]:
for p in process.keys() :
    my_entries[process[p]['label']] = []

In [207]:
for s in selection :
    my_entries["Selection"].append(selection[s])
    for p in process.keys() :
        my_entries[process[p]['label']].append(len(df[p][s]))

In [208]:
my_entries = pd.DataFrame(my_entries)
my_entries = my_entries.set_index('Selection')

In [209]:
my_entries

,$ttH \rightarrow \gamma\gamma$,$\gamma\gamma$ + jets,$tt\gamma\gamma$
Selection,,,
All events,306353,2470000,50000
$\geq 2$ $b$-jets,184750,53359,24265
$\geq 2$ photons,41319,19621,3854
120 < $m_{\gamma\gamma}$ < 130 GeV,39193,9689,454


In [210]:
my_yields = {
    "Selection" : []
}

In [211]:
for p in process.keys() :
    my_yields[process[p]['label']] = []

In [212]:
for s in selection :
    my_yields["Selection"].append(selection[s])
    for p in process.keys() :
        my_yields[process[p]['label']].append(df[p][s]["weight"].sum()*lumi)

In [213]:
my_yields = pd.DataFrame(my_yields)
my_yields = my_yields.set_index('Selection')

In [214]:
my_yields

,$ttH \rightarrow \gamma\gamma$,$\gamma\gamma$ + jets,$tt\gamma\gamma$
Selection,,,
All events,2.283668e+06,6.469200e+08,8.556000e+06
$\geq 2$ $b$-jets,1.377330e+06,1.397531e+07,4.152390e+06
$\geq 2$ photons,3.080275e+05,5.138954e+06,6.595221e+05
120 < $m_{\gamma\gamma}$ < 130 GeV,2.921770e+05,2.537655e+06,7.769174e+04
